# Preprocessing for Clustering

This notebook creates the first model-ready preprocessing output from the customer-level feature table. It does not run clustering or modelling.

## Imports

Use the reusable preprocessing functions from `src/preprocessing.py`.

In [ ]:
import sys

import pandas as pd

sys.path.append("../src")

from preprocessing import clean_feature_values, preprocess_for_clustering, select_model_features

## Load Customer Features

Load the combined customer-level feature table created during feature engineering.

In [ ]:
customer_features = pd.read_csv("../data/processed/customer_features.csv")
customer_features.head()

## Initial Checks

Check shape, duplicate customer IDs, and missing values before preprocessing.

In [ ]:
print(f"Rows before preprocessing: {customer_features.shape[0]:,}")
print(f"Columns before preprocessing: {customer_features.shape[1]:,}")
print(f"Duplicated customer_id before preprocessing: {customer_features['customer_id'].duplicated().sum():,}")

In [ ]:
missing_before = customer_features.isna().sum().sort_values(ascending=False)
missing_before[missing_before > 0]

## Clean and Select Features

Clean invalid and missing values, then remove non-model columns while keeping `customer_id` for traceability.

In [ ]:
cleaned_customer_features = clean_feature_values(customer_features)
selected_customer_features = select_model_features(cleaned_customer_features)

removed_columns = [
    column for column in customer_features.columns
    if column not in selected_customer_features.columns
]

removed_columns

In [ ]:
missing_after_cleaning = selected_customer_features.isna().sum().sort_values(ascending=False)
missing_after_cleaning[missing_after_cleaning > 0]

## Preprocess Model Features

One-hot encode categorical model columns and scale numeric model features with `StandardScaler`. `customer_id` is preserved and not used as a model feature.

In [ ]:
customer_features_model = preprocess_for_clustering(customer_features)
customer_features_model.head()

## Final Validation

Confirm the preprocessed output keeps all customers, has no duplicate IDs, and has no missing values in model features.

In [ ]:
preprocessing_validation = pd.DataFrame(
    {
        "check": [
            "rows before preprocessing",
            "rows after preprocessing",
            "columns before preprocessing",
            "columns after preprocessing",
            "duplicated customer_id after preprocessing",
            "missing values in model features",
        ],
        "value": [
            customer_features.shape[0],
            customer_features_model.shape[0],
            customer_features.shape[1],
            customer_features_model.shape[1],
            customer_features_model["customer_id"].duplicated().sum(),
            customer_features_model.drop(columns=["customer_id"]).isna().sum().sum(),
        ],
    }
)

preprocessing_validation

In [ ]:
missing_after = customer_features_model.drop(columns=["customer_id"]).isna().sum().sort_values(ascending=False)
missing_after[missing_after > 0]

In [ ]:
customer_features_model.columns.to_frame(index=False, name="feature_column")

## Save Preprocessed Feature Table

Save the preprocessed customer-level feature table. This file keeps `customer_id` for traceability, but clustering should use the remaining feature columns.

In [ ]:
customer_features_model.to_csv("../data/processed/customer_features_model.csv", index=False)
print("Saved ../data/processed/customer_features_model.csv")

## Preprocessing Notes

- `customer_id` is kept in the final file for traceability, but it is excluded from model feature scaling.
- `most_frequent_product` is removed because it is useful for profiling but not a simple numeric clustering feature.
- Numeric missing values are filled with the median, and categorical missing values are filled with `Unknown`.
- Invalid `percentage_of_products_bought_promotion` values are clipped to the `0` to `100` range.
- `customer_gender` and `degree_level` are one-hot encoded.
- Numeric model features are scaled with `StandardScaler`.
- No clustering or modelling is performed in this notebook.